# 📘 Notebook 5: Phase 5 cVAE Analysis

This notebook summarizes the Phase 5 conditional VAE outputs using the available evaluation report and training history, plus the 2026-07-24 advisor-requested experiment: weighted sampling by the v12 (auxetic) distribution.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "notebooks"))
import utils

root = Path.cwd()
phase5_dir = root/ ".." / "outputs" / "phase5"
phase3_dir = root / ".." / "outputs" / "phase3"
report_path = phase5_dir / "evaluation_report.json"
train_history_path = phase5_dir / "train_history.json"

print(f"Phase 5 output dir: {phase5_dir}")
if report_path.exists():
    with open(report_path, "r") as f:
        report = json.load(f)
    print("Loaded evaluation report")
    print(json.dumps(report, indent=2)[:2500])
else:
    print("evaluation_report.json not found")

if train_history_path.exists():
    with open(train_history_path, "r") as f:
        history = json.load(f)
    # train_history.json là 1 list các dict per-epoch (không phải dict) -
    # xem train.py::main() (json.dump(history_list, ...)).
    if isinstance(history, list):
        print(f"\nLoaded training history: {len(history)} epoch(s), "
              f"keys per epoch: {list(history[0].keys()) if history else '[]'}")
    else:
        print("\nLoaded training history keys:", list(history.keys())[:20])
else:
    print("train_history.json not found")

In [ ]:
# Show property accuracy summary
if 'report' in locals() and 'property_accuracy' in report:
    accuracy = report['property_accuracy']
    acc_df = pd.DataFrame([
        {"metric": k, "mae": v.get('mae'), "r2": v.get('r2')} for k, v in accuracy.items() if isinstance(v, dict)
    ])
    display(acc_df)

In [ ]:
# Plot basic diagnostics if present
if 'report' in locals() and 'diversity_check' in report:
    diversity = report['diversity_check']
    print('Diversity check condition:', diversity.get('condition'))
    print('Pixel std:', diversity.get('pixel_std'))

---
## Weighted Sampling theo phổ auxetic (advisor 2026-07-24)

> **⚠️ property_accuracy() ở trên đo qua surrogate CNN đóng băng — README cảnh báo con số này KHÔNG đáng tin (surrogate exploitation, xem [README §5](../README.md#5-conditional-vae-phase-5---thiết-kế-ngược-được-giải-quyết-qua-best-of-n--chọn-lọc-bằng-fe-thực)). Phần dưới đây dùng FE THẬT (`verify_fe.py`), không qua surrogate.**

Advisor: xem phổ phân bố v12 (đâu tập trung nhiều nhất), gán trọng số cho vùng thưa mẫu, lấy mẫu theo trọng số trong dataloader (`train.py --weighted-sampling`), train lại 1 lần xem thế nào. Chi tiết đầy đủ + diễn giải: [EXPERIMENT_LOG.md § Weighted sampling theo phổ auxetic](../EXPERIMENT_LOG.md#weighted-sampling-theo-phổ-auxetic-cho-phase-5-2026-07-24-yêu-cầu-advisor).

In [ ]:
# ── Phổ phân bố v12 + trọng số per-bin ──
weights_path = phase3_dir / "v12_bin_weights.json"
fig_path = phase5_dir / ".." / "figures" / "auxetic_properties" / "v12_distribution_weights.png"

if weights_path.exists():
    with open(weights_path) as f:
        w = json.load(f)
    edges = w["bin_edges"]
    counts = w["bin_count"]
    weight = w["bin_weight"]
    bin_df = pd.DataFrame({
        "bin_lo": edges[:-1], "bin_hi": edges[1:], "count": counts, "weight": weight,
    })
    bin_df["pct"] = (bin_df["count"] / bin_df["count"].sum() * 100).round(2)
    mode_row = bin_df.loc[bin_df["count"].idxmax()]
    print(f"Vùng tập trung nhiều nhất: [{mode_row.bin_lo:.2f}, {mode_row.bin_hi:.2f}) "
          f"({mode_row.pct:.1f}% số mẫu)")
    display(bin_df)
else:
    print("Chưa có v12_bin_weights.json - chạy "
          "'python3 analysis/scripts/analyze_auxetic_distribution.py' trước.")

if fig_path.exists():
    from PIL import Image
    img = Image.open(fig_path)
    plt.figure(figsize=(9, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.title("v12 distribution + per-bin sampling weight")
    plt.show()

In [ ]:
# ── So sánh checkpoint baseline vs weighted-sampling (FE thật) ──
comparison_path = phase5_dir / "weighted_sampling_comparison.json"

if comparison_path.exists():
    with open(comparison_path) as f:
        comparison = json.load(f)
    rows = []
    for ckpt_name, regions in comparison.items():
        for region_name, stats in regions.items():
            rows.append({
                "checkpoint": ckpt_name, "region": region_name,
                "mae_v12_fe": stats["mae_v12_fe"], "std": stats["std"], "n": stats["n"],
            })
    comp_df = pd.DataFrame(rows)
    display(comp_df.pivot(index="region", columns="checkpoint", values="mae_v12_fe").round(4))

    fig, ax = plt.subplots(figsize=(8, 5))
    pivot = comp_df.pivot(index="region", columns="checkpoint", values="mae_v12_fe")
    pivot.plot(kind="bar", ax=ax, color=["#4c72b0", "#c44e52"])
    ax.set_ylabel("MAE(v12, FE thật)")
    ax.set_title("Baseline vs Weighted-sampling — MAE theo vùng v12 (thấp hơn = tốt hơn)")
    ax.tick_params(axis="x", rotation=15)
    fig.tight_layout()
    plt.show()

    print("\nMAE thấp hơn = tốt hơn. Xem EXPERIMENT_LOG.md để biết diễn giải đầy đủ "
          "(bao gồm confound early-stopping đã gặp ở lần train đầu tiên).")
else:
    print("Chưa có weighted_sampling_comparison.json - chạy "
          "'python3 analysis/scripts/compare_weighted_sampling.py' sau khi train xong "
          "cvae_gamma20_weighted.pt.")